# Exercise 1 

In this exercise, we will work with the `spam.csv` data file. This file contains text and a label attached to each text. The goal is to clean the text data and then build classifcation models. 

### Exercise 1(a) (2 points)

Load the `pandas` and `numpy` libraries.

In [1]:
import pandas as pd
import numpy as np

### Exercise 1(b) (2 points)

Read the `csv` file and create a data-frame called `spam_df`.

In [2]:
spam_df = pd.read_csv('DATA/spam.csv')

### Exercise 1(c) (2 points)

Using the `string` library, remove all the punctuation of the `text` column.

In [3]:
import string

spam_df["text_clean"] = spam_df["text"].apply(
    lambda t: t.translate(str.maketrans("", "", string.punctuation))
)

### Exercise 1(d) (10 points)

Using the `nltk` library, remove stopwords and pronouns from `text_clean` and store in another column called `'text_clean_1'`. 

In [4]:
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

spam_df["text_clean_1"] = spam_df["text_clean"].apply(
    lambda t: " ".join(w for w in t.split() if w.lower() not in stop_words)
)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bdupey\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Exercise 1(e) (10 points)

Perform Lemmatization of `text_clean_1` and store the results in another column called `text_clean_2` in the `spam_df` data frame.

In [5]:
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet")
lemmatizer = WordNetLemmatizer()

spam_df["text_clean_2"] = spam_df["text_clean_1"].apply(
    lambda t: " ".join(lemmatizer.lemmatize(w) for w in t.split())
)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\bdupey\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


### Exercise 1(f) (5 points)

Based on the results from part 1(f), we see digits in some of the text in `text_clean_2`. Remove all the digits from `text_clean_2` and store the results in another column `text_clean_3` in the `spam_df` data frame.

In [6]:
spam_df["text_clean_3"] = spam_df["text_clean_2"].str.replace(r"\d+", "", regex=True)

### Exercise 1(g) (5 points)

Generate topic features using BERT embeddings on `text_clean_3`.

In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
bert_embeddings = model.encode(spam_df["text_clean_3"].tolist(), show_progress_bar=True)

Batches: 100%|██████████| 175/175 [00:04<00:00, 35.65it/s]


### Exercise 1(h) (5 points)

Apply the K-Means clustering algorithm to the bert embedding generated in the previous step.

1. Set the number of clusters to k=3.

2. Assign the cluster labels to a new column in `spam_df` called `'kmeans_cluster_label'`.

3. Generate new features by computing the distance of each document to the centroid of each of the k=3 clusters. Store these distances in a NumPy array called `X_kmeans_features`.

In [11]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
spam_df["kmeans_cluster_label"] = kmeans.fit_predict(bert_embeddings)
X_kmeans_features = kmeans.transform(bert_embeddings)

### Exercise 1(i) (5 points)

Hierarchical clustering is a type of density-based grouping used to organize documents based on similarity.

1. Apply Agglomerative Hierarchical Clustering to the L2​-normalized BERT embeddings (bert_embeddings) with k=3 clusters, using the `'ward'` linkage method.

2. Store the resulting cluster assignments in a new column called `'bert_hierarchical_label'` in `spam_df`.

3. Generate new features by computing the distance of each document to the centroid of each of the k=3 clusters. Store these distances in a NumPy array called `X_hierarchical_features`.

In [12]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import normalize

bert_norm = normalize(bert_embeddings)
hier = AgglomerativeClustering(n_clusters=3, linkage="ward")
spam_df["bert_hierarchical_label"] = hier.fit_predict(bert_norm)

centroids = np.array([bert_norm[spam_df["bert_hierarchical_label"] == k].mean(axis=0) for k in range(3)])
X_hierarchical_features = np.linalg.norm(bert_norm[:, None, :] - centroids[None, :, :], axis=2)

### Exercise 1(j) (10 points)

Train and evaluate six `LogisticRegression` models as follows:

- Using `kmeans` features as input.
- Using `hierarchical` features as input.
- Using bert embedding features as input.
- Combining `kmeans` + bert embeddings features as input.
- Combining `hierarchical` + bert embeddings features as input.
- Combining `kmeans` + `hierarchical` + bert embeddings features as input.

Evaluate the six models over a five-fold cross validation strategy.

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

y = spam_df["label"]

feature_sets = {
    "kmeans": X_kmeans_features,
    "hierarchical": X_hierarchical_features,
    "bert": bert_embeddings,
    "kmeans + bert": np.hstack([X_kmeans_features, bert_embeddings]),
    "hierarchical + bert": np.hstack([X_hierarchical_features, bert_embeddings]),
    "kmeans + hierarchical + bert": np.hstack([X_kmeans_features, X_hierarchical_features, bert_embeddings]),
}

for name, X in feature_sets.items():
    scores = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5)
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std():.4f})")

kmeans: 0.9130 (+/- 0.0039)
hierarchical: 0.9246 (+/- 0.0038)
bert: 0.9729 (+/- 0.0048)
kmeans + bert: 0.9731 (+/- 0.0044)
hierarchical + bert: 0.9733 (+/- 0.0040)
kmeans + hierarchical + bert: 0.9731 (+/- 0.0039)
